# Real-time Chord Detection with ONNX

This notebook loads a trained AlexNet chord classifier (ONNX format) and performs real-time chord prediction while playing audio.


In [1]:
import os
import json
import time
import threading
import numpy as np
import librosa
import onnxruntime as ort
from PIL import Image
import sounddevice as sd
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets


In [55]:
# Paths to model artifacts
version = 'v5'
MODEL_DIR = fr'results\{version}'
ONNX_MODEL_PATH = os.path.join(MODEL_DIR, 'alexnet_chord_classifier.onnx')
CONFIG_PATH = os.path.join(MODEL_DIR, 'chord_model_config.json')

# Load configuration
with open(CONFIG_PATH, 'r') as f:
    config = json.load(f)

print("Configuration loaded:")
print(f"  Sample rate: {config['sample_rate']} Hz")
print(f"  Duration: {config['duration']} seconds")
print(f"  Representation: {config['representation']}")
print(f"  Input size: {config['input_size']}")
print(f"  Number of classes: {config['num_classes']}")

# Load ONNX model
session = ort.InferenceSession(ONNX_MODEL_PATH)
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

print(f"\nONNX model loaded: {ONNX_MODEL_PATH}")
print(f"  Input name: {input_name}")
print(f"  Output name: {output_name}")


Configuration loaded:
  Sample rate: 44100 Hz
  Duration: 1 seconds
  Representation: cqt_chromagram
  Input size: [227, 227]
  Number of classes: 24

ONNX model loaded: results\v5\alexnet_chord_classifier.onnx
  Input name: input
  Output name: output


In [16]:
def preprocess_audio_chunk(audio_chunk, sr):
    """
    Preprocess an audio chunk for chord classification.
    Replicates the exact preprocessing from training.
    
    Args:
        audio_chunk: Audio signal (numpy array)
        sr: Sample rate
    
    Returns:
        Preprocessed tensor ready for ONNX inference (1, 3, 227, 227)
    """
    hop_length = config['hop_length']
    input_size = tuple(config['input_size'])
    normalize_mean = config['normalize_mean']
    normalize_std = config['normalize_std']
    
    # Compute CQT chromagram (matches training)
    chromagram = librosa.feature.chroma_cqt(
        y=audio_chunk, 
        sr=sr, 
        hop_length=hop_length,
        n_chroma=12,
        n_octaves=7,
    )
    
    # Convert to dB scale (matches training)
    chromagram_db = librosa.amplitude_to_db(chromagram + 1e-10, ref=np.max)
    
    # Normalize to 0-255 range
    norm_feature = (chromagram_db - chromagram_db.min()) / (chromagram_db.max() - chromagram_db.min() + 1e-10)
    norm_feature = (norm_feature * 255).astype(np.uint8)
    
    # Convert to PIL Image and resize
    img = Image.fromarray(norm_feature)
    img = img.convert("RGB")
    img = img.resize(input_size, Image.Resampling.BILINEAR)
    
    # Convert to numpy and normalize
    img_array = np.array(img).astype(np.float32) / 255.0
    
    # Apply normalization (mean=0.5, std=0.5 for each channel)
    for i in range(3):
        img_array[:, :, i] = (img_array[:, :, i] - normalize_mean[i]) / normalize_std[i]
    
    # Transpose to (C, H, W) and add batch dimension
    img_tensor = np.transpose(img_array, (2, 0, 1))
    img_tensor = np.expand_dims(img_tensor, axis=0)
    
    return img_tensor.astype(np.float32)


def predict_chord(audio_chunk, sr):
    """
    Predict chord from audio chunk.
    
    Returns:
        tuple: (predicted_chord, confidence, all_probabilities)
    """
    # Preprocess
    input_tensor = preprocess_audio_chunk(audio_chunk, sr)
    
    # Run inference
    outputs = session.run([output_name], {input_name: input_tensor})
    logits = outputs[0][0]
    
    # Apply softmax to get probabilities
    exp_logits = np.exp(logits - np.max(logits))
    probabilities = exp_logits / exp_logits.sum()
    
    # Get prediction
    predicted_idx = np.argmax(probabilities)
    confidence = probabilities[predicted_idx]
    predicted_chord = config['idx_to_class'][str(predicted_idx)]
    
    return predicted_chord, confidence, probabilities


print("Preprocessing functions defined.")

Preprocessing functions defined.


In [17]:
def play_and_predict(audio_path, window_duration=2.0, hop_duration=0.5):
    """
    Play audio and display real-time chord predictions.
    
    Args:
        audio_path: Path to the audio file
        window_duration: Duration of each analysis window (seconds)
        hop_duration: How often to update predictions (seconds)
    """
    sample_rate = config['sample_rate']
    
    # Load audio
    print(f"Loading audio: {audio_path}")
    y, sr = librosa.load(audio_path, sr=sample_rate)
    total_duration = len(y) / sr
    print(f"Audio duration: {total_duration:.2f} seconds")
    print(f"Sample rate: {sr} Hz")
    print("-" * 50)
    
    # Calculate window parameters
    window_samples = int(window_duration * sr)
    hop_samples = int(hop_duration * sr)
    
    # Shared state for threading
    playback_complete = threading.Event()
    current_time = [0.0]  # Use list to allow modification in nested function
    
    def audio_playback():
        """Play audio in background thread."""
        sd.play(y, sr)
        sd.wait()
        playback_complete.set()
    
    # Start audio playback in background
    playback_thread = threading.Thread(target=audio_playback)
    playback_thread.start()
    
    start_time = time.time()
    output_widget = widgets.Output()
    display(output_widget)
    
    try:
        while not playback_complete.is_set():
            # Calculate current position in audio
            elapsed = time.time() - start_time
            current_sample = int(elapsed * sr)
            
            # Get window of audio for prediction
            window_start = max(0, current_sample - window_samples // 2)
            window_end = min(len(y), window_start + window_samples)
            
            if window_end - window_start < sr * 0.5:  # Need at least 0.5s of audio
                time.sleep(0.1)
                continue
            
            audio_chunk = y[window_start:window_end]
            
            # Pad if too short
            if len(audio_chunk) < window_samples:
                audio_chunk = np.pad(audio_chunk, (0, window_samples - len(audio_chunk)), 'constant')
            
            # Predict chord
            chord, confidence, probs = predict_chord(audio_chunk, sr)
            
            # Get top 3 predictions
            top_indices = np.argsort(probs)[::-1][:3]
            top_chords = [(config['idx_to_class'][str(i)], probs[i]) for i in top_indices]
            
            # Format time display
            minutes = int(elapsed // 60)
            seconds = elapsed % 60
            total_min = int(total_duration // 60)
            total_sec = total_duration % 60
            
            # Create styled HTML output
            html_content = f"""
            <div style="font-family: 'Segoe UI', Arial, sans-serif; padding: 20px; background: linear-gradient(135deg, #1a1a2e 0%, #16213e 100%); border-radius: 15px; color: white; min-width: 400px;">
                <div style="text-align: center; margin-bottom: 20px;">
                    <span style="font-size: 14px; color: #888;">Now Playing</span>
                    <div style="font-size: 12px; color: #666; margin-top: 5px;">{os.path.basename(audio_path)}</div>
                </div>
                
                <div style="text-align: center; margin: 30px 0;">
                    <div style="font-size: 72px; font-weight: bold; color: #00d4ff; text-shadow: 0 0 20px rgba(0, 212, 255, 0.5);">
                        {chord}
                    </div>
                    <div style="font-size: 18px; color: #888; margin-top: 10px;">
                        Confidence: {confidence*100:.1f}%
                    </div>
                </div>
                
                <div style="background: rgba(255,255,255,0.1); border-radius: 10px; padding: 15px; margin: 20px 0;">
                    <div style="font-size: 12px; color: #888; margin-bottom: 10px;">Top Predictions:</div>
                    <div style="display: flex; justify-content: space-around;">
                        {"".join([f'<div style="text-align: center;"><div style="font-size: 24px; color: {"#00d4ff" if i == 0 else "#666"};">{c}</div><div style="font-size: 11px; color: #888;">{p*100:.1f}%</div></div>' for i, (c, p) in enumerate(top_chords)])}
                    </div>
                </div>
                
                <div style="text-align: center; margin-top: 20px;">
                    <span style="font-size: 24px; font-family: monospace;">{minutes:02d}:{seconds:05.2f}</span>
                    <span style="color: #666;"> / </span>
                    <span style="font-size: 16px; color: #888;">{total_min:02d}:{total_sec:05.2f}</span>
                </div>
                
                <div style="background: #333; height: 6px; border-radius: 3px; margin-top: 15px; overflow: hidden;">
                    <div style="background: linear-gradient(90deg, #00d4ff, #0099cc); height: 100%; width: {min(100, elapsed/total_duration*100):.1f}%; border-radius: 3px;"></div>
                </div>
            </div>
            """
            
            with output_widget:
                clear_output(wait=True)
                display(HTML(html_content))
            
            # Wait before next prediction
            time.sleep(hop_duration)
    
    except KeyboardInterrupt:
        sd.stop()
        print("\nPlayback stopped by user.")
    
    playback_thread.join()
    
    # Final display
    with output_widget:
        clear_output(wait=True)
        display(HTML("""
        <div style="font-family: 'Segoe UI', Arial, sans-serif; padding: 20px; background: linear-gradient(135deg, #1a1a2e 0%, #16213e 100%); border-radius: 15px; color: white; text-align: center;">
            <div style="font-size: 24px; color: #00d4ff; margin: 20px 0;">Playback Complete</div>
        </div>
        """))
    
    print("\nPlayback finished.")


print("Real-time playback function defined.")


Real-time playback function defined.


## Usage

Set the AUDIO_PATH variable below to your audio file and run the cell to start real-time chord detection.


In [6]:
# Set the path to your audio file here
import os
from random import randint

audios_path = r'C:\Users\Luis\Desktop\Master\DSAP\chord-detection-project\models\cnn_alexnet\test-audios'
test_audios = os.listdir(audios_path) 

#idx = randint(0, len(test_audios)-1)
idx = 0
AUDIO_PATH = os.path.join(audios_path, test_audios[idx]) # <-- Change this!

# Run real-time chord detection
# - window_duration: How much audio context to use for each prediction (default: 2.0 seconds)
# - hop_duration: How often to update the prediction (default: 0.5 seconds)
#play_and_predict(AUDIO_PATH, window_duration=config['duration'], hop_duration=0.5)


## Alternative: Quick Prediction (No Playback)

If you just want to predict chords without audio playback, use this function:


In [14]:
for i, audio in enumerate(test_audios):
    print(f'{i} - {audio}')
    

0 - [ A C#m Bm E ].mp3
1 - [ Am F Am G ].mp3
2 - [ C Am Dm G ].mp3
3 - [ C Am F G ] (1).mp3
4 - [ C Am F G ].mp3
5 - [ C Em Am F ].mp3
6 - [ Dm Am Bb F ].mp3
7 - [ E B C#m A ].mp3
8 - [ Eb Bb Cm Ab ].mp3
9 - [ Em C G D ].mp3
10 - [ F G Am Em ].mp3
11 - [ Gm Bb Eb F ].mp3


In [58]:
idx = 8
AUDIO_PATH = os.path.join(audios_path, test_audios[idx]) # <-- Change this!

In [59]:
def analyze_audio_file(audio_path, window_duration=2.0, hop_duration=1.0):
    """
    Analyze an entire audio file and print chord predictions at regular intervals.
    No audio playback - just prints predictions.
    
    Args:
        audio_path: Path to the audio file
        window_duration: Duration of each analysis window (seconds)
        hop_duration: Time between predictions (seconds)
    """
    sample_rate = config['sample_rate']
    
    # Load audio
    print(f"Loading audio: {audio_path}")
    y, sr = librosa.load(audio_path, sr=sample_rate)
    total_duration = len(y) / sr
    print(f"Audio duration: {total_duration:.2f} seconds")
    print("=" * 60)
    
    window_samples = int(window_duration * sr)
    hop_samples = int(hop_duration * sr)
    
    current_pos = 0
    results = []
    
    while current_pos + window_samples <= len(y):
        # Get audio chunk
        audio_chunk = y[current_pos:current_pos + window_samples]
        
        # Predict
        chord, confidence, _ = predict_chord(audio_chunk, sr)
        
        # Calculate timestamp
        timestamp = current_pos / sr
        
        results.append({
            'time': timestamp,
            'chord': chord,
            'confidence': confidence
        })
        
        print(f"[{timestamp:6.2f}s] {chord:4s} (confidence: {confidence*100:5.1f}%)")
        
        current_pos += hop_samples
    
    print("=" * 60)
    print(f"Analysis complete. Found {len(results)} chord segments.")
    
    return results


# Example usage (uncomment to run):
results = analyze_audio_file(AUDIO_PATH)


Loading audio: C:\Users\Luis\Desktop\Master\DSAP\chord-detection-project\models\cnn_alexnet\test-audios\[ Eb Bb Cm Ab ].mp3
Audio duration: 519.05 seconds
[  0.00s] D#   (confidence:  99.5%)
[  1.00s] D#   (confidence:  99.9%)
[  2.00s] D#   (confidence:  99.8%)
[  3.00s] D#   (confidence:  99.9%)
[  4.00s] D#   (confidence:  99.4%)
[  5.00s] Gm   (confidence:  36.9%)
[  6.00s] Gm   (confidence:  58.8%)
[  7.00s] Gm   (confidence:  62.9%)
[  8.00s] A#   (confidence:  81.8%)
[  9.00s] Gm   (confidence:  59.7%)
[ 10.00s] Cm   (confidence:  86.0%)
[ 11.00s] Cm   (confidence:  99.1%)
[ 12.00s] Cm   (confidence:  98.9%)
[ 13.00s] Cm   (confidence:  97.9%)
[ 14.00s] Cm   (confidence:  89.1%)
[ 15.00s] Fm   (confidence:  60.2%)
[ 16.00s] Fm   (confidence:  64.8%)
[ 17.00s] Fm   (confidence:  66.6%)
[ 18.00s] Fm   (confidence:  54.2%)
[ 19.00s] Cm   (confidence:  90.1%)
[ 20.00s] D#   (confidence:  97.6%)
[ 21.00s] D#   (confidence:  99.8%)
[ 22.00s] D#   (confidence:  99.8%)
[ 23.00s] D#   (c

KeyboardInterrupt: 

Stop any currently playing audio (useful if you need to interrupt)
sd.stop()

In [ ]:
sd.stop()